# 🏆 Baseline 1: EfficientNetB0 + MiniLM (Late Fusion)
## E-commerce Visual Search System — Shopee Dataset (34,250 items)

**Pipeline:**
- 🖼️ **Image Branch:** `EfficientNetB0` → 1280-dim features
- 📝 **Text Branch:** `paraphrase-multilingual-MiniLM-L12-v2` → 384-dim features
- 🔀 **Fusion:** `L2_Normalize(Concat([α * img, (1-α) * txt]))`
- 🔍 **Search:** FAISS `IndexFlatIP` (Cosine Similarity)
- 📊 **Metrics:** mAP@5, Precision@1, Recall@5

**Dataset Split:**
- Gallery: toàn bộ 34,250 ảnh
- Val queries (20%): ~6,850 ảnh → dùng để grid search alpha
- Test queries (80%): ~27,400 ảnh → đánh giá cuối cùng (chạy 1 lần)

## 📦 Cell 1: Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q faiss-gpu sentence-transformers torchvision Pillow pandas numpy tqdm scikit-learn
print('✅ Cài đặt thư viện hoàn tất!')

## 📂 Cell 2: Import thư viện & Cấu hình

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import faiss

# ─── CẤU HÌNH ───────────────────────────────────────────────────────────────
# 🔧 Chỉnh đường dẫn cho phù hợp với môi trường Colab của bạn
DATA_DIR    = '/kaggle/input/shopee-product-matching'  # Thay đổi nếu cần
CSV_PATH    = os.path.join(DATA_DIR, 'train.csv')
IMG_DIR     = os.path.join(DATA_DIR, 'train_images')

BATCH_SIZE  = 64
IMG_SIZE    = 224
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42

print(f'🔧 Thiết bị đang dùng: {DEVICE}')
print(f'📂 Thư mục dữ liệu: {DATA_DIR}')

## 📊 Cell 3: Đọc dữ liệu & Chia tập

In [ ]:
# Đọc file CSV
df = pd.read_csv(CSV_PATH)
print(f'📊 Tổng số mẫu: {len(df)}')
print(f'📋 Các cột: {list(df.columns)}')
print(df.head())

# ─── STRICT DATASET SPLITTING RULE ──────────────────────────────────────────
# Gallery = toàn bộ dataset (34,250 ảnh)
df_gallery = df.copy()
print(f'\n🗂️ Gallery size: {len(df_gallery)} ảnh')

# Chia val (20%) và test (80%) cho queries
# KHÔNG được dùng test để tuning — chỉ dùng val!
val_idx, test_idx = train_test_split(
    df.index.tolist(),
    test_size=0.8,
    random_state=RANDOM_SEED,
    stratify=df['label_group']
)

df_val  = df.loc[val_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)

print(f'✅ Val queries  (20%): {len(df_val)} ảnh  → dùng grid search alpha')
print(f'✅ Test queries (80%): {len(df_test)} ảnh → đánh giá cuối (chạy 1 lần!)')

## 🖼️ Cell 4: Trích xuất Image Features (EfficientNetB0)

In [ ]:
# Dataset cho ảnh
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'])
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img


# Transforms chuẩn ImageNet
img_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


def extract_image_features(df_input, model, transform, img_dir, batch_size=64):
    """Trích xuất image features từ EfficientNetB0."""
    dataset = ShopeeImageDataset(df_input, img_dir, transform)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)
    all_feats = []
    model.eval()
    with torch.no_grad():
        for imgs in tqdm(loader, desc='🖼️ Trích xuất image features'):
            imgs = imgs.to(DEVICE)
            feats = model(imgs)  # (B, 1280)
            all_feats.append(feats.cpu().numpy())
    return np.vstack(all_feats)  # (N, 1280)


# ─── Tải EfficientNetB0, bỏ classification head ─────────────────────────────
print('⏳ Đang tải mô hình EfficientNetB0...')
eff_net = models.efficientnet_b0(pretrained=True)
# Giữ lại tất cả trừ lớp cuối (classifier)
eff_net.classifier = torch.nn.Identity()
eff_net = eff_net.to(DEVICE)
eff_net.eval()
print('✅ EfficientNetB0 đã sẵn sàng (output dim: 1280)')

# Trích xuất features cho Gallery
print('\n📦 Đang trích xuất gallery image features...')
gallery_img_feats = extract_image_features(df_gallery, eff_net, img_transform, IMG_DIR, BATCH_SIZE)
print(f'✅ Gallery image features shape: {gallery_img_feats.shape}')

## 📝 Cell 5: Trích xuất Text Features (MiniLM)

In [ ]:
# Tải Sentence-Transformer MiniLM
print('⏳ Đang tải mô hình MiniLM...')
minilm = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=DEVICE)
print('✅ MiniLM đã sẵn sàng (output dim: 384)')


def extract_text_features(df_input, model, batch_size=512):
    """Trích xuất text features từ MiniLM."""
    titles = df_input['title'].fillna('').tolist()
    print(f'📝 Đang encode {len(titles)} câu tiêu đề...')
    feats = model.encode(
        titles,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False
    )
    return feats  # (N, 384)


# Trích xuất text features cho Gallery
print('\n📦 Đang trích xuất gallery text features...')
gallery_txt_feats = extract_text_features(df_gallery, minilm)
print(f'✅ Gallery text features shape: {gallery_txt_feats.shape}')

## 🔀 Cell 6: Late Fusion & FAISS Index

In [ ]:
def fuse_and_normalize(img_feats, txt_feats, alpha):
    """
    Late Fusion:
        fused = L2_Normalize(Concat([alpha * img, (1 - alpha) * txt]))
    """
    img_w = alpha * img_feats
    txt_w = (1 - alpha) * txt_feats
    fused = np.concatenate([img_w, txt_w], axis=1)  # (N, 1280 + 384)
    # L2 normalize từng vector
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-10, norms)  # tránh chia 0
    return (fused / norms).astype(np.float32)


def build_faiss_index(features):
    """Xây dựng FAISS IndexFlatIP cho Cosine Similarity (sau L2-normalize)."""
    dim = features.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(features)
    return index


print('✅ Hàm fuse_and_normalize và build_faiss_index đã sẵn sàng')

## 📐 Cell 7: Hàm đánh giá metrics

In [ ]:
def get_ground_truth_dict(df_input):
    """
    Tạo dict: posting_id → set(posting_id của các items cùng label_group)
    (bao gồm cả bản thân, sẽ loại sau khi search)
    """
    gt = {}
    for label, grp in df_input.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids:
            gt[pid] = ids
    return gt


def evaluate_retrieval(query_df, gallery_df, query_feats, gallery_feats,
                       alpha, K=5):
    """
    Đánh giá retrieval:
        - Fuse + normalize query & gallery
        - Build FAISS index trên gallery
        - Tìm K+1 nearest neighbors (loại self-match)
        - Tính mAP@K, P@1, Recall@K
    """
    # Fuse features
    q_fused = fuse_and_normalize(query_feats[0], query_feats[1], alpha)
    g_fused = fuse_and_normalize(gallery_feats[0], gallery_feats[1], alpha)

    # Ground truth cho toàn bộ gallery
    gt_dict = get_ground_truth_dict(gallery_df)

    # Build FAISS index
    index = build_faiss_index(g_fused)

    # Search: lấy K+1 để loại self-match
    scores, indices = index.search(q_fused, K + 1)

    gallery_pids = gallery_df['posting_id'].tolist()

    ap_list, p1_list, r5_list = [], [], []

    for i, row in enumerate(query_df.itertuples()):
        qid  = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}  # Loại bỏ self-match

        if len(relevant) == 0:
            continue

        # Lấy top-K kết quả (bỏ self-match)
        retrieved = []
        for idx in indices[i]:
            pid = gallery_pids[idx]
            if pid != qid:
                retrieved.append(pid)
            if len(retrieved) == K:
                break

        # Tính AP@K
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, start=1):
            if pid in relevant:
                hits += 1
                ap  += hits / rank
        ap /= min(len(relevant), K)
        ap_list.append(ap)

        # Tính P@1
        p1 = 1.0 if (retrieved and retrieved[0] in relevant) else 0.0
        p1_list.append(p1)

        # Tính Recall@K
        r5 = len(set(retrieved) & relevant) / len(relevant)
        r5_list.append(r5)

    return {
        'mAP@5':        np.mean(ap_list),
        'Precision@1':  np.mean(p1_list),
        'Recall@5':     np.mean(r5_list),
    }


print('✅ Hàm evaluate_retrieval đã sẵn sàng')

## 🔍 Cell 8: Trích xuất Query Features (Val & Test)

In [ ]:
# Trích xuất image & text features cho VAL queries
print('⏳ Đang trích xuất features cho VAL queries...')
val_img_feats = extract_image_features(df_val, eff_net, img_transform, IMG_DIR, BATCH_SIZE)
val_txt_feats = extract_text_features(df_val, minilm)
print(f'✅ Val img: {val_img_feats.shape}, txt: {val_txt_feats.shape}')

# Trích xuất image & text features cho TEST queries
print('\n⏳ Đang trích xuất features cho TEST queries...')
test_img_feats = extract_image_features(df_test, eff_net, img_transform, IMG_DIR, BATCH_SIZE)
test_txt_feats = extract_text_features(df_test, minilm)
print(f'✅ Test img: {test_img_feats.shape}, txt: {test_txt_feats.shape}')

## 🎯 Cell 9: Grid Search Alpha trên Validation Set

In [ ]:
# Grid search alpha từ 0.1 → 0.9 trên VAL queries
# KHÔNG được dùng test set ở bước này!

alphas = np.arange(0.1, 1.0, 0.1).round(1)
print(f'🔍 Grid search alpha: {alphas.tolist()}')
print('─' * 60)

val_results = []
best_alpha_1 = None
best_map5    = -1.0

for alpha in alphas:
    metrics = evaluate_retrieval(
        query_df     = df_val,
        gallery_df   = df_gallery,
        query_feats  = (val_img_feats, val_txt_feats),
        gallery_feats= (gallery_img_feats, gallery_txt_feats),
        alpha        = alpha,
        K            = 5
    )
    val_results.append({'alpha': alpha, **metrics})
    print(f'  alpha={alpha:.1f} | mAP@5={metrics["mAP@5"]:.4f} | '
          f'P@1={metrics["Precision@1"]:.4f} | R@5={metrics["Recall@5"]:.4f}')

    if metrics['mAP@5'] > best_map5:
        best_map5    = metrics['mAP@5']
        best_alpha_1 = alpha

print('─' * 60)
print(f'\n🏆 BEST_ALPHA_1 = {best_alpha_1:.1f}  (Val mAP@5 = {best_map5:.4f})')

## 🧪 Cell 10: Đánh giá cuối cùng trên TEST SET

In [ ]:
# ⚠️ CHỈ CHẠY 1 LẦN với BEST_ALPHA_1 tìm được từ val!
print(f'⚠️  Đang đánh giá TEST SET với alpha = {best_alpha_1:.1f}...')
print('⚠️  Lưu ý: bước này chỉ được chạy DUY NHẤT 1 LẦN!')

test_metrics_1 = evaluate_retrieval(
    query_df      = df_test,
    gallery_df    = df_gallery,
    query_feats   = (test_img_feats, test_txt_feats),
    gallery_feats = (gallery_img_feats, gallery_txt_feats),
    alpha         = best_alpha_1,
    K             = 5
)

print('\n📊 KẾT QUẢ TEST SET — Baseline 1 (EfficientNetB0 + MiniLM):')
print(f'   mAP@5        = {test_metrics_1["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_1["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_1["Recall@5"]:.4f}')

## 📋 Cell 11: Bảng kết quả Markdown (copy-paste vào báo cáo)

In [ ]:
# In bảng Markdown để dán vào báo cáo
from IPython.display import Markdown, display

feature_dim_1 = '1280 + 384'

table_md = f"""
## 📊 Kết quả so sánh Baseline Models — TEST SET

| Baseline Model | Feature Dim | Best Alpha | Test mAP@5 | Test Precision@1 | Test Recall@5 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| EfficientNetB0 + MiniLM | {feature_dim_1} | {best_alpha_1:.1f} | {test_metrics_1['mAP@5']:.4f} | {test_metrics_1['Precision@1']:.4f} | {test_metrics_1['Recall@5']:.4f} |
| MobileCLIP | — | — | — | — | — |
"""

display(Markdown(table_md))
print('\n📋 Raw Markdown (copy vào báo cáo):')
print(table_md)